In [14]:
import sys
sys.path.append('./scriprts')  # Ensure custom transformers can be found if needed

import streamlit as st
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

st.set_page_config(page_title="Vaccine Prediction & Insights", layout="wide")
st.title("🦠 Vaccine Prediction & Insights Dashboard")

@st.cache_resource
def load_artifacts():
    pipeline = joblib.load("ML_pipeline.pkl")
    xgb_h1n1 = joblib.load("models/XGBoost_learning_rate-0.1_max_depth-5_n_estimators-100_h1n1_vaccine.joblib")
    xgb_seasonal = joblib.load("models/XGBoost_learning_rate-0.1_max_depth-3_n_estimators-200_seasonal_vaccine.joblib")
    fi_h1n1 = pd.read_csv("feature_importance_h1n1_vaccine.csv")
    fi_seasonal = pd.read_csv("feature_importance_seasonal_vaccine.csv")
    train_features = pd.read_csv("data/training_set_features.csv")
    train_labels = pd.read_csv("data/training_set_labels.csv")
    return pipeline, xgb_h1n1, xgb_seasonal, fi_h1n1, fi_seasonal, train_features, train_labels

pipeline, xgb_h1n1, xgb_seasonal, fi_h1n1, fi_seasonal, train_features, train_labels = load_artifacts()

tab1, tab2 = st.tabs(["Make a Prediction", "Dataset Dashboard"])

# =========================
# TAB 1: PREDICTION
# =========================
with tab1:
    st.header("Predict H1N1 or Seasonal Flu Vaccination")
    vaccine_type = st.radio(
        "Which vaccine do you want to predict?",
        ["h1n1_vaccine", "seasonal_vaccine"],
        format_func=lambda x: "H1N1 Vaccine" if x == "h1n1_vaccine" else "Seasonal Flu Vaccine"
    )

    user_input = {}
    # Required fields (including those missing previously)
    user_input['respondent_id'] = 0  # Hidden, but required by pipeline
    user_input['h1n1_concern'] = st.slider("H1N1 Concern (0=Not at all concerned, 3=Very concerned)", 0, 3, 1)
    user_input['h1n1_knowledge'] = st.slider("H1N1 Knowledge (0=No knowledge, 2=High knowledge)", 0, 2, 1)

    # Demographics
    user_input['age_group'] = st.selectbox("Age Group", [
        '18 - 34 Years', '35 - 44 Years', '45 - 54 Years', '55 - 64 Years', '65+ Years'])
    user_input['education'] = st.selectbox("Education", [
        '< 12 Years', '12 Years', 'Some College', 'College Graduate'])
    user_input['race'] = st.selectbox("Race", [
        'White', 'Black', 'Hispanic', 'Other or Multiple'])
    user_input['sex'] = st.selectbox("Sex", ['Male', 'Female'])
    user_input['income_poverty'] = st.selectbox("Household Income", [
        'Below Poverty', '<= $75,000, Above Poverty', '> $75,000'])
    user_input['marital_status'] = st.selectbox("Marital Status", [
        'Married', 'Not Married'])
    user_input['rent_or_own'] = st.selectbox("Rent or Own", [
        'Own', 'Rent', 'Other or Unknown'])
    user_input['employment_status'] = st.selectbox("Employment Status", [
        'Employed', 'Not in Labor Force', 'Unemployed'])
    user_input['employment_industry'] = st.selectbox("Employment Industry", [
        'advisory', 'arts', 'business', 'clerical', 'construction', 'education', 'engineering',
        'healthcare', 'hospitality', 'insurance', 'manufacturing', 'other', 'real_estate',
        'retail', 'self_employed', 'services', 'transportation', 'Not in Labor Force', 'nan'])
    user_input['employment_occupation'] = st.selectbox("Employment Occupation", [
        'artists', 'clerical', 'cleaning', 'construction', 'education', 'engineering', 'executive',
        'healthcare', 'hospitality', 'lawyer', 'management', 'manual', 'office', 'other', 'real_estate',
        'retail', 'sales', 'science', 'self_employed', 'services', 'transportation', 'Not in Labor Force', 'nan'])
    user_input['hhs_geo_region'] = st.selectbox("HHS Geo Region", [
        'region_1', 'region_2', 'region_3', 'region_4', 'region_5', 'region_6', 'region_7', 'region_8', 'region_9', 'region_10'])
    user_input['census_msa'] = st.selectbox("Census MSA", [
        'MSA, Not Principle  City', 'MSA, Principle City', 'Non-MSA'])

    # Household
    user_input['household_adults'] = st.number_input("Number of Adults in Household", 0, 10, 2)
    user_input['household_children'] = st.number_input("Number of Children in Household", 0, 10, 0)

    # Doctor recommendations
    user_input['doctor_recc_h1n1'] = st.selectbox("Doctor Recommended H1N1 Vaccine?", [0, 1])
    user_input['doctor_recc_seasonal'] = st.selectbox("Doctor Recommended Seasonal Vaccine?", [0, 1])

    # Health/behavioral
    user_input['chronic_med_condition'] = st.selectbox("Chronic Medical Condition?", [0, 1])
    user_input['child_under_6_months'] = st.selectbox("Child Under 6 Months in Household?", [0, 1])
    user_input['health_worker'] = st.selectbox("Health Worker?", [0, 1])
    user_input['health_insurance'] = st.selectbox("Health Insurance?", [0, 1])

    # Behavioral variables
    user_input['opinion_h1n1_vacc_effective'] = st.slider("Opinion: H1N1 Vaccine Effective (0=Strongly Disagree, 4=Strongly Agree)", 0, 4, 2)
    user_input['opinion_h1n1_risk'] = st.slider("Opinion: H1N1 Risk (0=Low, 4=High)", 0, 4, 2)
    user_input['opinion_h1n1_sick_from_vacc'] = st.slider("Opinion: H1N1 Sick from Vaccine (0=Strongly Disagree, 4=Strongly Agree)", 0, 4, 2)
    user_input['opinion_seas_vacc_effective'] = st.slider("Opinion: Seasonal Vaccine Effective (0=Strongly Disagree, 4=Strongly Agree)", 0, 4, 2)
    user_input['opinion_seas_risk'] = st.slider("Opinion: Seasonal Risk (0=Low, 4=High)", 0, 4, 2)
    user_input['opinion_seas_sick_from_vacc'] = st.slider("Opinion: Seasonal Sick from Vaccine (0=Strongly Disagree, 4=Strongly Agree)", 0, 4, 2)

    # Behavioral (binary)
    user_input['behavioral_antiviral_meds'] = st.selectbox("Used Antiviral Meds?", [0, 1])
    user_input['behavioral_avoidance'] = st.selectbox("Avoided Contact with Others?", [0, 1])
    user_input['behavioral_face_mask'] = st.selectbox("Used Face Mask?", [0, 1])
    user_input['behavioral_wash_hands'] = st.selectbox("Washed Hands Frequently?", [0, 1])
    user_input['behavioral_large_gatherings'] = st.selectbox("Avoided Large Gatherings?", [0, 1])
    user_input['behavioral_outside_home'] = st.selectbox("Reduced Time Outside Home?", [0, 1])
    user_input['behavioral_touch_face'] = st.selectbox("Touched Face Less?", [0, 1])

    input_df = pd.DataFrame([user_input])

    if st.button("Predict"):
        X_proc = pipeline.transform(input_df)
        if vaccine_type == "h1n1_vaccine":
            pred = xgb_h1n1.predict(X_proc)[0]
            proba = xgb_h1n1.predict_proba(X_proc)[0,1]
        else:
            pred = xgb_seasonal.predict(X_proc)[0]
            proba = xgb_seasonal.predict_proba(X_proc)[0,1]
        st.success(f"Prediction: {'Will Take Vaccine' if pred == 1 else 'Will NOT Take Vaccine'}")
        st.info(f"Predicted probability: {proba:.2%}")

# =========================
# TAB 2: DASHBOARD (EQUAL COLUMNS, MEANINGFUL VISUALS)
# =========================
with tab2:
    st.header("Dataset & Model Insights")

    # Three equal columns for visualizations
    col1, col2, col3 = st.columns(3)

    # Feature Importance
    with col1:
        st.subheader("Feature Importance")
        tab_choice = st.radio("Model", ["H1N1 Vaccine", "Seasonal Flu Vaccine"], horizontal=True, key="fi_radio")
        fi = fi_h1n1 if tab_choice == "H1N1 Vaccine" else fi_seasonal
        st.caption(f"Top features for {tab_choice} prediction")
        top_n = st.slider("Top N features", 5, min(20, len(fi)), 10, key="fi_slider")
        fig, ax = plt.subplots(figsize=(6, 0.4*top_n + 1))
        plot_df = fi.sort_values("importance", ascending=False).head(top_n)
        ax.barh(plot_df['feature'][::-1], plot_df['importance'][::-1], color="skyblue")
        ax.set_xlabel("Importance")
        ax.set_ylabel("Feature")
        st.pyplot(fig)

    # Class Distribution
    with col2:
        st.subheader("Class Distribution")
        h1n1_counts = train_labels['h1n1_vaccine'].value_counts().sort_index()
        seasonal_counts = train_labels['seasonal_vaccine'].value_counts().sort_index()
        df_counts = pd.DataFrame({
            'H1N1 Vaccine': h1n1_counts,
            'Seasonal Vaccine': seasonal_counts
        }).fillna(0)
        df_counts.index = ['No', 'Yes']
        df_counts.plot(kind='bar', ax=plt.gca(), color=['#3498db', '#e67e22'])
        plt.ylabel("Count")
        plt.title("Vaccine Uptake")
        plt.xticks(rotation=0)
        st.pyplot(plt.gcf())
        plt.clf()

    # Correlation Heatmap
    with col3:
        st.subheader("Correlation Heatmap")
        numeric_cols = [
            'household_adults', 'household_children',
            'opinion_h1n1_vacc_effective', 'opinion_h1n1_risk', 'opinion_h1n1_sick_from_vacc',
            'opinion_seas_vacc_effective', 'opinion_seas_risk', 'opinion_seas_sick_from_vacc'
        ]
        corr_df = train_features[numeric_cols].corr()
        fig, ax = plt.subplots(figsize=(6, 5))
        sns.heatmap(corr_df, annot=True, cmap="coolwarm", ax=ax)
        st.pyplot(fig)

    st.markdown("---")
    # Feature Distribution Plots (full width)
    st.subheader("Feature Distribution")
    feat = st.selectbox("Select a feature to view its distribution", train_features.columns)
    fig, ax = plt.subplots(figsize=(10, 3))
    if train_features[feat].dtype == 'object':
        train_features[feat].value_counts().plot.bar(ax=ax, color="mediumorchid")
        ax.set_ylabel("Count")
    else:
        train_features[feat].plot.hist(ax=ax, bins=20, color="mediumorchid")
        ax.set_ylabel("Frequency")
    ax.set_title(f"Distribution of {feat}")
    st.pyplot(fig)

    st.markdown("---")
    # Raw Data Preview (full width)
    st.subheader("Raw Data Preview")
    st.dataframe(train_features.head(20), use_container_width=True)

    st.caption("Explore more: Add your own EDA, plots, or metrics as needed!")


2025-06-18 19:29:31.458 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-06-18 19:29:31.459 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-06-18 19:29:31.460 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-06-18 19:29:31.460 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-06-18 19:29:31.462 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-06-18 19:29:31.463 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-06-18 19:29:31.464 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-06-18 19:29:31.465 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [15]:
!streamlit run h1n1seasonal.py

^C


In [16]:
%%writefile test2.py
import sys
sys.path.append('./scriprts')  # Only needed if your pipeline uses custom code

import streamlit as st
import pandas as pd
import numpy as np
import joblib

st.title("H1N1/Seasonal Vaccine Prediction")

@st.cache_resource
def load_artifacts():
    pipeline = joblib.load("ML_pipeline.pkl")
    xgb_h1n1 = joblib.load("models/XGBoost_learning_rate-0.1_max_depth-5_n_estimators-100_h1n1_vaccine.joblib")
    xgb_seasonal = joblib.load("models/XGBoost_learning_rate-0.1_max_depth-3_n_estimators-200_seasonal_vaccine.joblib")
    train_features = pd.read_csv("data/training_set_features.csv")
    return pipeline, xgb_h1n1, xgb_seasonal, train_features

pipeline, xgb_h1n1, xgb_seasonal, train_features = load_artifacts()

required_columns = list(train_features.columns)

with st.form("prediction_form"):
    st.header("Enter your information")

    # Interactive fields (expand as you wish)
    user_input = {}
    user_input['h1n1_concern'] = st.slider("H1N1 Concern (0-3)", 0, 3, 1)
    user_input['h1n1_knowledge'] = st.slider("H1N1 Knowledge (0-2)", 0, 2, 1)
    user_input['age_group'] = st.selectbox("Age Group", [
        '18 - 34 Years', '35 - 44 Years', '45 - 54 Years', '55 - 64 Years', '65+ Years'])
    user_input['sex'] = st.selectbox("Sex", ['Male', 'Female'])
    user_input['opinion_h1n1_vacc_effective'] = st.slider("Opinion: H1N1 Vaccine Effective (0-4)", 0, 4, 2)

    # Fill the rest with defaults
    for col in required_columns:
        if col not in user_input:
            if train_features[col].dtype == 'O':
                user_input[col] = train_features[col].mode()[0]
            elif "id" in col:
                user_input[col] = 0
            else:
                user_input[col] = int(train_features[col].mode()[0])  # Use mode for categorical/int, 0 for respondent_id

    vaccine_type = st.radio(
        "Which vaccine do you want to predict?",
        ["h1n1_vaccine", "seasonal_vaccine"],
        format_func=lambda x: "H1N1 Vaccine" if x == "h1n1_vaccine" else "Seasonal Flu Vaccine"
    )

    submitted = st.form_submit_button("Predict")

    if submitted:
        input_df = pd.DataFrame([user_input])[required_columns]
        X_proc = pipeline.transform(input_df)
        if vaccine_type == "h1n1_vaccine":
            pred = xgb_h1n1.predict(X_proc)[0]
            proba = xgb_h1n1.predict_proba(X_proc)[0,1]
        else:
            pred = xgb_seasonal.predict(X_proc)[0]
            proba = xgb_seasonal.predict_proba(X_proc)[0,1]
        st.success(f"Prediction: {'Will Take Vaccine' if pred == 1 else 'Will NOT Take Vaccine'}")
        st.info(f"Predicted probability: {proba:.2%}")


Writing test2.py


In [17]:
!streamlit run test2.py

^C


In [57]:
%%writefile test1.py
import streamlit as st
import pandas as pd
import joblib
import random
import plotly.express as px

# Load pipeline and models
preprocessor = joblib.load("ML_pipeline.pkl")
model_h1n1 = joblib.load("models/XGBoost_learning_rate-0.1_max_depth-5_n_estimators-100_h1n1_vaccine.joblib")
model_seasonal = joblib.load("models/XGBoost_learning_rate-0.1_max_depth-3_n_estimators-200_seasonal_vaccine.joblib")

# Load data for dashboard
train_features = pd.read_csv("data/training_set_features.csv")
train_labels = pd.read_csv("data/training_set_labels.csv")
data = pd.merge(train_features, train_labels, on="respondent_id")

tab1, tab2 = st.tabs(["🧪 Prediction Model", "📊 Dataset Dashboard"])

with tab1:
    st.title("Flu Vaccine Prediction App")
    st.write("Answer the following questions to predict your likelihood of having received the H1N1 and seasonal flu vaccines.")

    def user_input_features():
        user_input = {}

        user_input['respondent_id'] = random.randint(100000, 999999)

        st.header("🦠 H1N1 Perceptions")
        user_input['h1n1_concern'] = st.slider("H1N1 Concern", 0, 3, 1, format="%d")
        user_input['h1n1_knowledge'] = st.slider("H1N1 Knowledge", 0, 2, 1, format="%d")

        st.header("👤 Demographics")
        user_input['age_group'] = st.selectbox("Age Group", [
            '18 - 34 Years', '35 - 44 Years', '45 - 54 Years', '55 - 64 Years', '65+ Years'])
        user_input['education'] = st.selectbox("Education", [
            '< 12 Years', '12 Years', 'Some College', 'College Graduate'])
        user_input['race'] = st.selectbox("Race", [
            'White', 'Black', 'Hispanic', 'Other or Multiple'])
        user_input['sex'] = st.selectbox("Sex", ['Male', 'Female'])
        user_input['income_poverty'] = st.selectbox("Household Income", [
            'Below Poverty', '<= $75,000, Above Poverty', '> $75,000'])
        user_input['marital_status'] = st.selectbox("Marital Status", [
            'Married', 'Not Married'])
        user_input['rent_or_own'] = st.selectbox("Rent or Own", [
            'Own', 'Rent', 'Other or Unknown'])
        user_input['employment_status'] = st.selectbox("Employment Status", [
            'Employed', 'Not in Labor Force', 'Unemployed'])
        user_input['employment_industry'] = st.selectbox("Employment Industry", [
            'advisory', 'arts', 'business', 'clerical', 'construction', 'education', 'engineering',
            'healthcare', 'hospitality', 'insurance', 'manufacturing', 'other', 'real_estate',
            'retail', 'self_employed', 'services', 'transportation', 'Not in Labor Force', 'nan'])
        user_input['employment_occupation'] = st.selectbox("Employment Occupation", [
            'artists', 'clerical', 'cleaning', 'construction', 'education', 'engineering', 'executive',
            'healthcare', 'hospitality', 'lawyer', 'management', 'manual', 'office', 'other', 'real_estate',
            'retail', 'sales', 'science', 'self_employed', 'services', 'transportation', 'Not in Labor Force', 'nan'])
        user_input['hhs_geo_region'] = st.selectbox("HHS Geo Region", [
            'region_1', 'region_2', 'region_3', 'region_4', 'region_5', 'region_6', 'region_7', 'region_8', 'region_9', 'region_10'])
        user_input['census_msa'] = st.selectbox("Census MSA", [
            'MSA, Not Principle City', 'MSA, Principle City', 'Non-MSA'])

        st.header("🏠 Household")
        user_input['household_adults'] = st.number_input("Number of Adults in Household", 0, 10, 2)
        user_input['household_children'] = st.number_input("Number of Children in Household", 0, 10, 0)

        st.header("🩺 Doctor Recommendations")
        user_input['doctor_recc_h1n1'] = st.selectbox("Doctor Recommended H1N1 Vaccine?", [0, 1])
        user_input['doctor_recc_seasonal'] = st.selectbox("Doctor Recommended Seasonal Vaccine?", [0, 1])

        st.header("🏥 Health & Behavioral")
        user_input['chronic_med_condition'] = st.selectbox("Chronic Medical Condition?", [0, 1])
        user_input['child_under_6_months'] = st.selectbox("Child Under 6 Months in Household?", [0, 1])
        user_input['health_worker'] = st.selectbox("Health Worker?", [0, 1])
        user_input['health_insurance'] = st.selectbox("Health Insurance?", [0, 1])

        st.header("💬 Opinions")
        user_input['opinion_h1n1_vacc_effective'] = st.slider("H1N1 Vaccine Effective (0=Strongly Disagree, 4=Strongly Agree)", 0, 4, 2, format="%d")
        user_input['opinion_h1n1_risk'] = st.slider("H1N1 Risk (0=Low, 4=High)", 0, 4, 2, format="%d")
        user_input['opinion_h1n1_sick_from_vacc'] = st.slider("H1N1 Sick from Vaccine (0=Strongly Disagree, 4=Strongly Agree)", 0, 4, 2, format="%d")
        user_input['opinion_seas_vacc_effective'] = st.slider("Seasonal Vaccine Effective (0=Strongly Disagree, 4=Strongly Agree)", 0, 4, 2, format="%d")
        user_input['opinion_seas_risk'] = st.slider("Seasonal Risk (0=Low, 4=High)", 0, 4, 2, format="%d")
        user_input['opinion_seas_sick_from_vacc'] = st.slider("Seasonal Sick from Vaccine (0=Strongly Disagree, 4=Strongly Agree)", 0, 4, 2, format="%d")

        st.header("🧼 Behaviors")
        user_input['behavioral_antiviral_meds'] = st.selectbox("Used Antiviral Meds?", [0, 1])
        user_input['behavioral_avoidance'] = st.selectbox("Avoided Contact with Others?", [0, 1])
        user_input['behavioral_face_mask'] = st.selectbox("Used Face Mask?", [0, 1])
        user_input['behavioral_wash_hands'] = st.selectbox("Washed Hands Frequently?", [0, 1])
        user_input['behavioral_large_gatherings'] = st.selectbox("Avoided Large Gatherings?", [0, 1])
        user_input['behavioral_outside_home'] = st.selectbox("Reduced Time Outside Home?", [0, 1])
        user_input['behavioral_touch_face'] = st.selectbox("Touched Face Less?", [0, 1])

        return pd.DataFrame([user_input])

    input_df = user_input_features()
    cols = ['respondent_id'] + [col for col in input_df.columns if col != 'respondent_id']
    input_df = input_df[cols]

    float_cols = [
        'h1n1_concern', 'h1n1_knowledge', 'behavioral_antiviral_meds', 'behavioral_avoidance',
        'behavioral_face_mask', 'behavioral_wash_hands', 'behavioral_large_gatherings',
        'behavioral_outside_home', 'behavioral_touch_face', 'doctor_recc_h1n1',
        'doctor_recc_seasonal', 'chronic_med_condition', 'child_under_6_months',
        'health_worker', 'health_insurance', 'opinion_h1n1_vacc_effective',
        'opinion_h1n1_risk', 'opinion_h1n1_sick_from_vacc', 'opinion_seas_vacc_effective',
        'opinion_seas_risk', 'opinion_seas_sick_from_vacc', 'household_adults', 'household_children'
    ]
    for col in float_cols:
        input_df[col] = input_df[col].astype(float)
    input_df['respondent_id'] = input_df['respondent_id'].astype(int)

    if st.button("Predict"):
        X = preprocessor.transform(input_df)
        h1n1_prob = model_h1n1.predict_proba(X)[:, 1][0]
        seasonal_prob = model_seasonal.predict_proba(X)[:, 1][0]

        st.subheader("Prediction Results")
        st.write(f"Probability you received the **H1N1 vaccine**: {h1n1_prob:.2%}")
        st.write(f"Probability you received the **seasonal flu vaccine**: {seasonal_prob:.2%}")

        st.info("These probabilities are based on your answers and the model's predictions.")


with tab2:
    st.title("📊 Dataset Dashboard")
    st.write("Explore key insights from the dataset.")

    # Vaccine uptake rates
    st.subheader("Vaccine Uptake Rates")
    h1n1_rate = data['h1n1_vaccine'].mean()
    seasonal_rate = data['seasonal_vaccine'].mean()
    st.metric("H1N1 Vaccine Uptake", f"{h1n1_rate:.1%}")
    st.metric("Seasonal Vaccine Uptake", f"{seasonal_rate:.1%}")

    # Age group distribution
    st.subheader("Age Group Distribution")
    st.bar_chart(data['age_group'].value_counts())

    # H1N1 vaccine by age group
    st.subheader("H1N1 Vaccine by Age Group")
    h1n1_by_age = data.groupby('age_group')['h1n1_vaccine'].mean().sort_index()
    st.bar_chart(h1n1_by_age)

    # Seasonal vaccine by age group
    st.subheader("Seasonal Vaccine by Age Group")
    seasonal_by_age = data.groupby('age_group')['seasonal_vaccine'].mean().sort_index()
    st.bar_chart(seasonal_by_age)

    # Vaccine uptake by sex
    st.subheader("Vaccine Uptake by Sex")
    sex_vax = data.groupby('sex')[['h1n1_vaccine', 'seasonal_vaccine']].mean()
    st.bar_chart(sex_vax)

    # Vaccine uptake by education level
    st.subheader("Vaccine Uptake by Education Level")
    edu_vax = data.groupby('education')[['h1n1_vaccine', 'seasonal_vaccine']].mean().sort_index()
    st.bar_chart(edu_vax)

    # Vaccine uptake by income level
    st.subheader("Vaccine Uptake by Income Level")
    income_vax = data.groupby('income_poverty')[['h1n1_vaccine', 'seasonal_vaccine']].mean()
    st.bar_chart(income_vax)

    # Chronic medical condition distribution
    st.subheader("Chronic Medical Condition Distribution")
    chronic_dist = data['chronic_med_condition'].value_counts().sort_index()
    chronic_dist.index = chronic_dist.index.map({0: "No", 1: "Yes"})
    st.bar_chart(chronic_dist)

    # Interactive: H1N1 vaccine uptake by age group and sex (Plotly grouped bar chart)
    st.subheader("H1N1 Vaccine Uptake by Age Group and Sex")
    fig = px.bar(
        data, 
        x='age_group', 
        y='h1n1_vaccine', 
        color='sex',
        barmode='group',
        labels={'h1n1_vaccine': 'H1N1 Vaccine Uptake'},
        title="H1N1 Vaccine Uptake by Age Group and Sex"
    )
    st.plotly_chart(fig)

    # Optional: Interactive filtering example
    st.subheader("Filter Vaccine Uptake by Age Group")
    selected_age_group = st.selectbox("Select Age Group", data['age_group'].unique())
    filtered = data[data['age_group'] == selected_age_group]
    uptake_by_sex = filtered.groupby('sex')[['h1n1_vaccine', 'seasonal_vaccine']].mean()
    st.bar_chart(uptake_by_sex)

    # Pie chart: Health insurance coverage
    st.subheader("Health Insurance Coverage")
    fig_ins = px.pie(
        data, 
        names=data['health_insurance'].map({0: "No", 1: "Yes"}),
        title="Proportion with Health Insurance"
    )
    st.plotly_chart(fig_ins)

    st.write("You can expand this dashboard with more charts and insights as needed.")


Overwriting test1.py


In [58]:
!streamlit run test1.py

^C


In [46]:
!pip install plotly

   ---------------------------------------- 0.0/16.3 MB ? eta -:--:--
   --------- ------------------------------ 3.9/16.3 MB 21.3 MB/s eta 0:00:01
   ------------------------------ --------- 12.3/16.3 MB 30.8 MB/s eta 0:00:01
   ---------------------------------------- 16.3/16.3 MB 33.0 MB/s eta 0:00:00


In [13]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib
import random
import plotly.express as px

# Add your main app title here
st.title("Flu Vaccine Prediction & Insights Dashboard 🦠💉")

# Load pipeline and models
preprocessor = joblib.load("ML_pipeline.pkl")
model_h1n1 = joblib.load("models/XGBoost_learning_rate-0.1_max_depth-5_n_estimators-100_h1n1_vaccine.joblib")
model_seasonal = joblib.load("models/XGBoost_learning_rate-0.1_max_depth-3_n_estimators-200_seasonal_vaccine.joblib")

# Load data for dashboard
train_features = pd.read_csv("data/training_set_features.csv")
train_labels = pd.read_csv("data/training_set_labels.csv")
data = pd.merge(train_features, train_labels, on="respondent_id")

tab1, tab2 = st.tabs(["🧪 Prediction Model", "📊 Dataset Dashboard"])

with tab1:
    st.title("Flu Vaccine Prediction App")
    st.write("Answer the following questions to predict your likelihood of having received the H1N1 and seasonal flu vaccines.")

    def user_input_features():
        user_input = {}

        user_input['respondent_id'] = random.randint(100000, 999999)

        st.header("🦠 H1N1 Perceptions")
        user_input['h1n1_concern'] = st.slider("H1N1 Concern", 0, 3, 1, format="%d")
        user_input['h1n1_knowledge'] = st.slider("H1N1 Knowledge", 0, 2, 1, format="%d")

        st.header("👤 Demographics")
        user_input['age_group'] = st.selectbox("Age Group", [
            '18 - 34 Years', '35 - 44 Years', '45 - 54 Years', '55 - 64 Years', '65+ Years'])
        user_input['education'] = st.selectbox("Education", [
            '< 12 Years', '12 Years', 'Some College', 'College Graduate'])
        user_input['race'] = st.selectbox("Race", [
            'White', 'Black', 'Hispanic', 'Other or Multiple'])
        user_input['sex'] = st.selectbox("Sex", ['Male', 'Female'])
        user_input['income_poverty'] = st.selectbox("Household Income", [
            'Below Poverty', '<= $75,000, Above Poverty', '> $75,000'])
        user_input['marital_status'] = st.selectbox("Marital Status", [
            'Married', 'Not Married'])
        user_input['rent_or_own'] = st.selectbox("Rent or Own", [
            'Own', 'Rent', 'Other or Unknown'])
        user_input['employment_status'] = st.selectbox("Employment Status", [
            'Employed', 'Not in Labor Force', 'Unemployed'])
        user_input['employment_industry'] = st.selectbox("Employment Industry", [
            'advisory', 'arts', 'business', 'clerical', 'construction', 'education', 'engineering',
            'healthcare', 'hospitality', 'insurance', 'manufacturing', 'other', 'real_estate',
            'retail', 'self_employed', 'services', 'transportation', 'Not in Labor Force', 'nan'])
        user_input['employment_occupation'] = st.selectbox("Employment Occupation", [
            'artists', 'clerical', 'cleaning', 'construction', 'education', 'engineering', 'executive',
            'healthcare', 'hospitality', 'lawyer', 'management', 'manual', 'office', 'other', 'real_estate',
            'retail', 'sales', 'science', 'self_employed', 'services', 'transportation', 'Not in Labor Force', 'nan'])
        user_input['hhs_geo_region'] = st.selectbox("HHS Geo Region", [
            'region_1', 'region_2', 'region_3', 'region_4', 'region_5', 'region_6', 'region_7', 'region_8', 'region_9', 'region_10'])
        user_input['census_msa'] = st.selectbox("Census MSA", [
            'MSA, Not Principle City', 'MSA, Principle City', 'Non-MSA'])

        st.header("🏠 Household")
        user_input['household_adults'] = st.number_input("Number of Adults in Household", 0, 10, 2)
        user_input['household_children'] = st.number_input("Number of Children in Household", 0, 10, 0)

        st.header("🩺 Doctor Recommendations")
        user_input['doctor_recc_h1n1'] = st.selectbox("Doctor Recommended H1N1 Vaccine?", [0, 1])
        user_input['doctor_recc_seasonal'] = st.selectbox("Doctor Recommended Seasonal Vaccine?", [0, 1])

        st.header("🏥 Health & Behavioral")
        user_input['chronic_med_condition'] = st.selectbox("Chronic Medical Condition?", [0, 1])
        user_input['child_under_6_months'] = st.selectbox("Child Under 6 Months in Household?", [0, 1])
        user_input['health_worker'] = st.selectbox("Health Worker?", [0, 1])
        user_input['health_insurance'] = st.selectbox("Health Insurance?", [0, 1])

        st.header("💬 Opinions")
        user_input['opinion_h1n1_vacc_effective'] = st.slider("H1N1 Vaccine Effective (0=Strongly Disagree, 4=Strongly Agree)", 0, 4, 2, format="%d")
        user_input['opinion_h1n1_risk'] = st.slider("H1N1 Risk (0=Low, 4=High)", 0, 4, 2, format="%d")
        user_input['opinion_h1n1_sick_from_vacc'] = st.slider("H1N1 Sick from Vaccine (0=Strongly Disagree, 4=Strongly Agree)", 0, 4, 2, format="%d")
        user_input['opinion_seas_vacc_effective'] = st.slider("Seasonal Vaccine Effective (0=Strongly Disagree, 4=Strongly Agree)", 0, 4, 2, format="%d")
        user_input['opinion_seas_risk'] = st.slider("Seasonal Risk (0=Low, 4=High)", 0, 4, 2, format="%d")
        user_input['opinion_seas_sick_from_vacc'] = st.slider("Seasonal Sick from Vaccine (0=Strongly Disagree, 4=Strongly Agree)", 0, 4, 2, format="%d")

        st.header("🧼 Behaviors")
        user_input['behavioral_antiviral_meds'] = st.selectbox("Used Antiviral Meds?", [0, 1])
        user_input['behavioral_avoidance'] = st.selectbox("Avoided Contact with Others?", [0, 1])
        user_input['behavioral_face_mask'] = st.selectbox("Used Face Mask?", [0, 1])
        user_input['behavioral_wash_hands'] = st.selectbox("Washed Hands Frequently?", [0, 1])
        user_input['behavioral_large_gatherings'] = st.selectbox("Avoided Large Gatherings?", [0, 1])
        user_input['behavioral_outside_home'] = st.selectbox("Reduced Time Outside Home?", [0, 1])
        user_input['behavioral_touch_face'] = st.selectbox("Touched Face Less?", [0, 1])

        return pd.DataFrame([user_input])

    input_df = user_input_features()
    cols = ['respondent_id'] + [col for col in input_df.columns if col != 'respondent_id']
    input_df = input_df[cols]

    float_cols = [
        'h1n1_concern', 'h1n1_knowledge', 'behavioral_antiviral_meds', 'behavioral_avoidance',
        'behavioral_face_mask', 'behavioral_wash_hands', 'behavioral_large_gatherings',
        'behavioral_outside_home', 'behavioral_touch_face', 'doctor_recc_h1n1',
        'doctor_recc_seasonal', 'chronic_med_condition', 'child_under_6_months',
        'health_worker', 'health_insurance', 'opinion_h1n1_vacc_effective',
        'opinion_h1n1_risk', 'opinion_h1n1_sick_from_vacc', 'opinion_seas_vacc_effective',
        'opinion_seas_risk', 'opinion_seas_sick_from_vacc', 'household_adults', 'household_children'
    ]
    for col in float_cols:
        input_df[col] = input_df[col].astype(float)
    input_df['respondent_id'] = input_df['respondent_id'].astype(int)

    if st.button("Predict"):
        X = preprocessor.transform(input_df)
        h1n1_prob = model_h1n1.predict_proba(X)[:, 1][0]
        seasonal_prob = model_seasonal.predict_proba(X)[:, 1][0]

        st.subheader("Prediction Results")
        st.write(f"Probability you received the **H1N1 vaccine**: {h1n1_prob:.2%}")
        st.write(f"Probability you received the **seasonal flu vaccine**: {seasonal_prob:.2%}")

        st.info("These probabilities are based on your answers and the model's predictions.")

with tab2:
    st.title("📊 Dataset Dashboard")
    st.write("Explore key insights from the dataset.")

    # Vaccine uptake rates
    st.subheader("Vaccine Uptake Rates")
    h1n1_rate = data['h1n1_vaccine'].mean()
    seasonal_rate = data['seasonal_vaccine'].mean()
    st.metric("H1N1 Vaccine Uptake", f"{h1n1_rate:.1%}")
    st.metric("Seasonal Vaccine Uptake", f"{seasonal_rate:.1%}")

    # Age group distribution
    st.subheader("Age Group Distribution")
    st.bar_chart(data['age_group'].value_counts())

    # H1N1 vaccine by age group
    st.subheader("H1N1 Vaccine by Age Group")
    h1n1_by_age = data.groupby('age_group')['h1n1_vaccine'].mean().sort_index()
    st.bar_chart(h1n1_by_age)

    # Seasonal vaccine by age group
    st.subheader("Seasonal Vaccine by Age Group")
    seasonal_by_age = data.groupby('age_group')['seasonal_vaccine'].mean().sort_index()
    st.bar_chart(seasonal_by_age)

    # Vaccine uptake by sex
    st.subheader("Vaccine Uptake by Sex")
    sex_vax = data.groupby('sex')[['h1n1_vaccine', 'seasonal_vaccine']].mean()
    st.bar_chart(sex_vax)

    # Vaccine uptake by education level
    st.subheader("Vaccine Uptake by Education Level")
    edu_vax = data.groupby('education')[['h1n1_vaccine', 'seasonal_vaccine']].mean().sort_index()
    st.bar_chart(edu_vax)

    # Vaccine uptake by income level
    st.subheader("Vaccine Uptake by Income Level")
    income_vax = data.groupby('income_poverty')[['h1n1_vaccine', 'seasonal_vaccine']].mean()
    st.bar_chart(income_vax)

    # Chronic medical condition distribution
    st.subheader("Chronic Medical Condition Distribution")
    chronic_dist = data['chronic_med_condition'].value_counts().sort_index()
    chronic_dist.index = chronic_dist.index.map({0: "No", 1: "Yes"})
    st.bar_chart(chronic_dist)

    # Interactive: H1N1 vaccine uptake by age group and sex (Plotly grouped bar chart)
    st.subheader("H1N1 Vaccine Uptake by Age Group and Sex")
    fig = px.bar(
        data, 
        x='age_group', 
        y='h1n1_vaccine', 
        color='sex',
        barmode='group',
        labels={'h1n1_vaccine': 'H1N1 Vaccine Uptake'},
        title="H1N1 Vaccine Uptake by Age Group and Sex"
    )
    st.plotly_chart(fig)

    # Optional: Interactive filtering example
    st.subheader("Filter Vaccine Uptake by Age Group")
    selected_age_group = st.selectbox("Select Age Group", data['age_group'].unique())
    filtered = data[data['age_group'] == selected_age_group]
    uptake_by_sex = filtered.groupby('sex')[['h1n1_vaccine', 'seasonal_vaccine']].mean()
    st.bar_chart(uptake_by_sex)

    # Pie chart: Health insurance coverage
    st.subheader("Health Insurance Coverage")
    fig_ins = px.pie(
        data, 
        names=data['health_insurance'].map({0: "No", 1: "Yes"}),
        title="Proportion with Health Insurance"
    )
    st.plotly_chart(fig_ins)

    # Feature importance plot (if available)
    st.subheader("Feature Importance: H1N1 Vaccine Model")
    try:
        fi_h1n1 = pd.read_csv("feature_importance_h1n1_vaccine.csv")
        # Only show top 10 features for clarity
        top_fi = fi_h1n1.sort_values(by="importance", ascending=False).head(10)
        fig_fi = px.bar(
            top_fi,
            x="importance",
            y="feature",
            orientation="h",
            title="Top 10 Features (H1N1 Vaccine Model)",
            labels={"importance": "Importance", "feature": "Feature"},
            height=400,
        )
        fig_fi.update_layout(yaxis={'categoryorder':'total ascending'})
        st.plotly_chart(fig_fi, use_container_width=True)
    except Exception as e:
        st.info("Feature importance file for H1N1 vaccine not found or invalid.")


Overwriting app.py


In [14]:
!streamlit run app.py

^C
